# Unified Lab 6: Strategic Interactions & Opponent Modeling

## The Scenario
**You are the Lead AI Architect at "AeroLogistics."** You are no longer just fighting gravity; you are fighting the free market. A rival delivery corporation, "SkyDyne," has deployed its own fleet in the exact same metropolitan airspace.

Every hour, both companies must simultaneously select one of three main flight corridors (Altitude A, Altitude B, or Altitude C) for their heavy drone traffic. Due to complex wind drafts and wake turbulence, this creates a dynamic competitive game (mathematically identical to Rock-Paper-Scissors):
* **Altitude A** gains a speed advantage over **Altitude B**.
* **Altitude B** gains a speed advantage over **Altitude C**.
* **Altitude C** gains a speed advantage over **Altitude A**.
* If both choose the same altitude, it causes air-traffic congestion and both lose time.

You need to build an AI that doesn't just learn a static environment, but actively models and exploits SkyDyne's routing strategy.

## Milestone 1 (The Baseline): The Historical Frequency Model (Fictitious Play)
**Objective:** Build an Opponent Modeler that tracks the rival's past actions to estimate their underlying probability distribution, forming the foundation of empirical strategy prediction.


### The Architect's Blueprint

Instruct your AI to build an agent class (`OpponentModelingAgent`) focused purely on observation and prediction (we will add the decision-making logic in the next milestone). Give the AI these specific constraints:
* **Action Space:** There are 3 discrete actions (0, 1, 2).
* **The Tracker:** The agent must maintain an internal count of how many times the opponent has played each specific action over the course of the interaction.
* **The Predictor (`get_opponent_probabilities`):** This method must return a probability distribution array by dividing the count of each action by the total number of actions observed. (Make sure the AI handles the edge case of step 0, where no actions have been observed yet, by returning a uniform distribution: `[1/3, 1/3, 1/3]`).
* Instantiate the class as `modeling_agent` so it can be evaluated by the Test Harness.

In [ ]:
# ENTER YOUR CODE HERE

### The Test Harness

*Use this exact block of code; your AI's generated code MUST pass these assertions.*

In [ ]:
# MILESTONE 1 TEST HARNESS - DO NOT MODIFY

import numpy as np

# Test 1: Ensure the agent has the correct tracking methods
assert hasattr(modeling_agent, 'update_opponent_history'), "Agent must have a method to record opponent actions."
assert hasattr(modeling_agent, 'get_opponent_probabilities'), "Agent must have a method to return predicted probabilities."

# Test 2: Simulate observing the opponent
# Opponent plays: A(0), A(0), B(1), A(0)
modeling_agent.update_opponent_history(0)
modeling_agent.update_opponent_history(0)
modeling_agent.update_opponent_history(1)
modeling_agent.update_opponent_history(0)

# Test 3: Verify the predicted probability distribution
probs = modeling_agent.get_opponent_probabilities()
assert len(probs) == 3, "Probabilities must cover all 3 possible altitudes."
assert round(sum(probs), 5) == 1.0, "Probabilities must sum to 1.0."
assert probs[0] == 0.75, "Agent should predict a 75% chance of the opponent choosing Altitude A based on history."
assert probs[1] == 0.25, "Agent should predict a 25% chance of the opponent choosing Altitude B."
assert probs[2] == 0.0, "Agent should predict a 0% chance of the opponent choosing Altitude C."

### Architect's Audit (Milestone 1)

1. Look at the underlying logic of your `get_opponent_probabilities` method. It treats all historical observations with equal weight. Imagine the rival drone's AI is programmed to play Action 0 for the first 1,000 rounds, but then suddenly switches to playing Action 1 exclusively. Mathematically, what is the flaw in relying on an unweighted historical average, and how long (in rounds) will it take for your agent to even predict a $50\%$ chance that the opponent plays Action 1?

ENTER YOUR ANSWERS HERE

## Milestone 2 (The Best Response): Exploiting the Model
**Objective:** Use the probability distribution generated by your Opponent Modeler to calculate the expected value of every possible action, and systematically choose the action that maximizes your payoff.

### The Architect's Blueprint

Instruct your AI to build a `BestResponseAgent` class (it can inherit from your Milestone 1 agent or be a new class). Give the AI these specific constraints:
* **The Payoff Matrix:** Define the environment's rules mathematically as a 2D array/list. Rows represent your agent's action; Columns represent the opponent's action.
    * Win = `+1`, Lose = `-1`, Tie (Congestion) = `-1`.
    * Remember: Action 0 beats 1; Action 1 beats 2; Action 2 beats 0.
* **The Math (`get_expected_values`):** Create a method that takes the opponent's probability distribution as input. It must multiply the payoffs for each of your actions by the probability of the opponent playing the corresponding response, summing them up to find the Expected Value (EV) of playing Action 0, 1, and 2.
* **The Policy (`get_best_response`):** Create a method that evaluates the expected values and returns the integer of the action with the highest EV (e.g., using `np.argmax`).
* Instantiate the class as `best_response_agent` so it can be evaluated by the Test Harness.

In [ ]:
# ENTER YOUR CODE HERE

### The Test Harness

*Use this exact block of code; your AI's generated code MUST pass these assertions.*


In [ ]:
# MILESTONE 2 TEST HARNESS - DO NOT MODIFY

# Test 1: Verify the expected value calculation
# Assume opponent has an 80% chance to play 0, 20% to play 1, 0% to play 2.
dummy_probs = [0.8, 0.2, 0.0]
expected_values = best_response_agent.get_expected_values(dummy_probs)

assert len(expected_values) == 3, "Must calculate an expected value for all 3 actions."
# Action 2 beats Action 0 (+1). 80% of the time we get +1, 20% we get -1 (Action 2 loses to Action 1). EV = 0.6
assert round(expected_values[2], 5) == 0.6, "Expected value for Action 2 is mathematically incorrect."

# Test 2: Verify the best response selection
best_action = best_response_agent.get_best_response(dummy_probs)
assert best_action == 2, "Agent failed to select the mathematically optimal action."

### Architect's Audit (Milestone 2)

1. Pass a perfectly uniform distribution (`[0.333, 0.333, 0.333]`) into your `get_expected_values` function. Look at the resulting expected values. Because they are mathematically identical, how does your policy function (like `np.argmax`) break the tie? If your agent defaults to a hardcoded tie-breaker (e.g., always picking Action 0 when EVs are equal), explain mechanically how "SkyDyne" (the rival AI) could easily exploit your agent, even though your agent thinks it is playing "optimally" against a random opponent.

ENTER YOUR ANSWERS HERE

## Milestone 3 (The Upgrade): Adaptive Opponent Modeling

**Objective:** Overcome the inertia of standard Fictitious Play by implementing an Exponentially Weighted Moving Average (EWMA). This allows your agent to "forget" the distant past and rapidly adapt to sudden shifts in the rival's strategy.


### The Architect's Blueprint

Instruct your AI to build an `AdaptiveModelingAgent` class that predicts the opponent's moves, but discards the simple "counting" method from Milestone 1. Give the AI these specific constraints:
* **The Tracker:** Instead of counting total historical actions, the agent directly maintains its predicted probability distribution `opponent_probs` (initialized to `[1/3, 1/3, 1/3]`).
* **The Math:** Implement an update rule using a decay factor (learning rate) $\alpha$. When the opponent plays an action, update the distribution: $P_{new} = (1 - \alpha) \times P_{old} + \alpha \times \text{Observed Action (as a one-hot array)}$.
* **Hyperparameter:** Set $\alpha = 0.2$.
* Instantiate the class as `adaptive_agent` so it can be evaluated by the Test Harness.
* **The Simulation:** Write a loop that simulates 100 rounds. For the first 50 rounds, the opponent plays Action 0 exclusively. For the next 50 rounds, the opponent plays Action 1 exclusively. Feed this identical data to *both* your Milestone 1 agent and your new Milestone 3 agent. Plot their predicted probability of Action 1 over time on the same graph.

In [ ]:
# ENTER YOUR CODE HERE

### The Test Harness

*Use this exact block of code; your AI's generated code MUST pass these assertions.*


In [ ]:
# MILESTONE 3 TEST HARNESS - DO NOT MODIFY

# Test 1: Verify initialization and update structure
assert hasattr(adaptive_agent, 'alpha'), "Agent must have a learning rate/decay factor (alpha)."
assert hasattr(adaptive_agent, 'opponent_probs'), "Agent must maintain an internal probability distribution."

# Test 2: Simulate the exact scenario from the Milestone 1 Audit
# Opponent plays Action 0 for a long time, then suddenly switches to Action 1.
# Let's start the agent with an artificial history where it heavily believes the opponent plays Action 0.
adaptive_agent.opponent_probs = np.array([0.99, 0.01, 0.0])

# Observe Action 1 just TWO times.
adaptive_agent.update_opponent_history(1)
adaptive_agent.update_opponent_history(1)

# Test 3: Verify rapid adaptation (Assuming an alpha of 0.2)
probs = adaptive_agent.get_opponent_probabilities()
assert round(sum(probs), 5) == 1.0, "Probabilities must still sum to 1.0."
assert probs[1] > 0.35, "Agent should have adapted much faster than the unweighted model."

### Architect's Audit (Milestone 3)

1. Examine the plot comparing the two agents. Look specifically at round 50, where the opponent suddenly switches to Action 1. How many rounds does it take for the Milestone 1 agent to predict a $>50\%$ chance of Action 1? How many rounds does it take for the Milestone 3 agent? Mathematically, what happens to the "weight" of the opponent's round 1 behavior by the time your adaptive agent reaches round 60?

ENTER YOUR ANSWERS HERE

## The Summary Audit

To complete the lab, submit your generated notebook along with brief, 3-4 sentence answers to the following questions:

1. **Data-Binding (The Memory Tradeoff):** You set your $\alpha$ value to $0.2$, which allowed the agent to adapt quickly. What would happen to your agent's predictive stability if you set $\alpha = 0.99$? Refer specifically to the mathematical update rule ($P_{new} = (1 - \alpha) \times P_{old} + \dots$) to explain why a very high $\alpha$ might make your agent too reactive to a single random mistake made by the opponent.
2. **The AI Consultation & Application:** Ask your preferred AI assistant: *"In Game Theory, what is the Nash Equilibrium strategy for a standard game of Rock-Paper-Scissors?"* Read the AI's explanation. Then, mathematically trace what happens if SkyDyne (the opponent) perfectly executes this Nash Equilibrium strategy against your `BestResponseAgent` from Milestone 2. If SkyDyne's true distribution is perfectly uniform, what specific Expected Value will your agent calculate for all three of its actions, and why does this mean your agent can no longer gain an exploitative advantage?

ENTER YOUR ANSWERS HERE